# Capítulo 3: Clustering de Textos

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caracena/apunte-analitica-textual/blob/main/capitulos/clase3-clustering-textos.ipynb)

## Objetivos de aprendizaje

- Comprender los fundamentos del aprendizaje no supervisado aplicado a texto.
- Implementar el algoritmo K-Means para agrupar documentos.
- Evaluar la calidad de los clusters obtenidos.
- Aplicar técnicas de reducción de dimensionalidad para visualización.

## 3.1 ¿Qué es el Clustering de Textos?

El **clustering** (agrupamiento) es una técnica de aprendizaje **no supervisado** que organiza documentos en grupos (*clusters*) basándose en su similitud, sin necesidad de etiquetas predefinidas.

### Aplicaciones

- **Descubrimiento de temas**: Identificar los temas principales en un corpus.
- **Organización de documentos**: Agrupar archivos similares automáticamente.
- **Detección de duplicados**: Encontrar documentos casi idénticos.
- **Segmentación de clientes**: Agrupar clientes según sus comentarios o reseñas.
- **Exploración de datos**: Entender la estructura de un corpus antes del análisis.

## 3.2 El algoritmo K-Means

**K-Means** es uno de los algoritmos de clustering más populares por su simplicidad y eficiencia.

### Algoritmo

1. Elegir $K$ (número de clusters).
2. Inicializar $K$ centroides aleatoriamente.
3. **Asignación**: Cada documento se asigna al centroide más cercano.
4. **Actualización**: Recalcular cada centroide como el promedio de los documentos asignados.
5. Repetir pasos 3-4 hasta convergencia.

### Función objetivo

K-Means minimiza la **inercia** (suma de distancias al cuadrado de cada punto a su centroide):

$$J = \sum_{k=1}^{K} \sum_{x_i \in C_k} \|x_i - \mu_k\|^2$$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize

# Corpus de ejemplo con temas variados
corpus = [
    # Tema: Deportes
    "El equipo de fútbol ganó el campeonato nacional tras una gran temporada",
    "El tenista chileno avanzó a la final del torneo internacional",
    "La selección de básquetbol se prepara para los juegos panamericanos",
    "El maratón de Santiago reunió a más de 30 mil corredores este año",
    # Tema: Tecnología
    "La nueva versión del sistema operativo incluye mejoras de seguridad",
    "La inteligencia artificial está transformando la industria del software",
    "El procesador de última generación ofrece mayor rendimiento y eficiencia",
    "La empresa tecnológica presentó su nuevo teléfono inteligente",
    # Tema: Salud
    "El ministerio de salud anunció una nueva campaña de vacunación masiva",
    "Los investigadores descubrieron un tratamiento prometedor contra el cáncer",
    "La alimentación saludable reduce el riesgo de enfermedades cardiovasculares",
    "El hospital inauguró una nueva unidad de cuidados intensivos"
]

etiquetas_reales = ["Deportes"] * 4 + ["Tecnología"] * 4 + ["Salud"] * 4

In [ ]:
# Vectorizar con TF-IDF
tfidf = TfidfVectorizer(max_features=100)
X = tfidf.fit_transform(corpus)

print(f"Dimensiones de la matriz TF-IDF: {X.shape}")
print(f"  - {X.shape[0]} documentos")
print(f"  - {X.shape[1]} términos en el vocabulario")

In [ ]:
# Aplicar K-Means
K = 3
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
X_normalized = normalize(X)
clusters = kmeans.fit_predict(X_normalized)

# Mostrar resultados
print("Asignación de clusters:\n")
for i, (doc, cluster, real) in enumerate(zip(corpus, clusters, etiquetas_reales)):
    print(f"  Cluster {cluster} ({real}): {doc[:60]}...")

## 3.3 Selección del número de clusters

### Método del codo (*Elbow Method*)

Consiste en ejecutar K-Means con distintos valores de $K$ y graficar la inercia. El "codo" indica el punto donde agregar más clusters no mejora significativamente.

### Coeficiente de Silhouette

Mide qué tan bien separados están los clusters. Valores entre -1 y 1:
- Cercano a 1: Los clusters están bien separados.
- Cercano a 0: Los clusters se solapan.
- Negativo: Puntos asignados al cluster incorrecto.

In [ ]:
# Método del codo y Silhouette
rango_k = range(2, 8)
inercias = []
silhouettes = []

for k in rango_k:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_normalized)
    inercias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_normalized, km.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(rango_k, inercias, 'bo-')
ax1.set_xlabel('Número de clusters (K)')
ax1.set_ylabel('Inercia')
ax1.set_title('Método del Codo')
ax1.grid(True, alpha=0.3)

ax2.plot(rango_k, silhouettes, 'rs-')
ax2.set_xlabel('Número de clusters (K)')
ax2.set_ylabel('Coeficiente de Silhouette')
ax2.set_title('Análisis de Silhouette')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mejor K según Silhouette: {rango_k[np.argmax(silhouettes)]}")

## 3.4 Visualización de clusters

Usamos **PCA** (Análisis de Componentes Principales) para reducir la dimensionalidad y poder visualizar los clusters en 2D.

In [ ]:
# Reducir a 2D con PCA
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_normalized.toarray())

# Visualizar
fig, ax = plt.subplots(figsize=(8, 6))
colores = ['#e74c3c', '#3498db', '#2ecc71']
nombres_clusters = [f'Cluster {i}' for i in range(K)]

for k in range(K):
    mask = clusters == k
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               c=colores[k], label=nombres_clusters[k], s=100, alpha=0.7)

# Agregar etiquetas
for i, txt in enumerate(etiquetas_reales):
    ax.annotate(txt, (X_2d[i, 0], X_2d[i, 1]),
                fontsize=8, ha='center', va='bottom')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} varianza)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} varianza)')
ax.set_title('Clusters de documentos (PCA 2D)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3.5 Interpretación de clusters: Palabras clave

Para entender qué representa cada cluster, podemos extraer los términos más relevantes de cada centroide.

In [ ]:
# Palabras clave por cluster
terminos = tfidf.get_feature_names_out()

print("Palabras clave por cluster:\n")
for k in range(K):
    centroide = kmeans.cluster_centers_[k]
    indices_top = centroide.argsort()[::-1][:5]
    palabras_top = [terminos[i] for i in indices_top]
    print(f"  Cluster {k}: {', '.join(palabras_top)}")

## 3.6 Otros algoritmos de clustering

| Algoritmo | Ventajas | Desventajas |
|-----------|----------|-------------|
| **K-Means** | Rápido, escalable | Requiere definir K, sensible a inicialización |
| **DBSCAN** | No requiere K, detecta outliers | Sensible a parámetros eps y min_samples |
| **Jerárquico** | Genera dendrograma, no requiere K | Costoso computacionalmente |


In [ ]:
from sklearn.cluster import DBSCAN, AgglomerativeClustering

# DBSCAN
dbscan = DBSCAN(eps=1.0, min_samples=2)
clusters_dbscan = dbscan.fit_predict(X.toarray())
print(f"DBSCAN encontró {len(set(clusters_dbscan)) - (1 if -1 in clusters_dbscan else 0)} clusters")
print(f"  Puntos ruidosos: {(clusters_dbscan == -1).sum()}")

# Clustering Jerárquico
agg = AgglomerativeClustering(n_clusters=3)
clusters_agg = agg.fit_predict(X.toarray())
print(f"\nClustering Jerárquico - Silhouette: {silhouette_score(X, clusters_agg):.3f}")
print(f"K-Means - Silhouette: {silhouette_score(X, clusters):.3f}")

## 3.7 Ejemplo

Usando los datos del repositorio https://github.com/jpposadas/FakeNewsCorpusSpanish vamos a realizar un análisis de clustering.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize
import numpy as np
import requests


# 1. Descargar y cargar el archivo Excel
df = pd.read_excel("train.xlsx")

df['Text'] = df['Text'].fillna('') # Rellenar posibles NaN en la columna de texto
documents = df['Text'].astype(str).tolist()

# 2. Preprocesar y vectorizar con TF-IDF
# Aumentar max_features y añadir stop_words para un corpus en español
data_stopwords = requests.get("https://raw.githubusercontent.com/stopwords-iso/stopwords-es/refs/heads/master/stopwords-es.txt")
stopwords = data_stopwords.text.split("\n")

tfidf_vectorizer = TfidfVectorizer(max_features=600, stop_words=stopwords)
X_df = tfidf_vectorizer.fit_transform(documents)
X_df_normalized = normalize(X_df)

print(f"Dimensiones de la matriz TF-IDF del corpus FakeNews: {X_df.shape}")
print(f"  - {X_df.shape[0]} documentos")
print(f"  - {X_df.shape[1]} términos en el vocabulario\n")


In [ ]:
# 3. Selección del número de clusters (K)
rango_k = range(2, 11) # Probar un rango de K de 2 a 10
inercias = []
silhouettes = []

for k in rango_k:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_df_normalized)
    inercias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_df_normalized, km.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(rango_k, inercias, 'bo-')
ax1.set_xlabel('Número de clusters (K)')
ax1.set_ylabel('Inercia')
ax1.set_title('Método del Codo (Corpus FakeNews)')
ax1.grid(True, alpha=0.3)

ax2.plot(rango_k, silhouettes, 'rs-')
ax2.set_xlabel('Número de clusters (K)')
ax2.set_ylabel('Coeficiente de Silhouette')
ax2.set_title('Análisis de Silhouette (Corpus FakeNews)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Determinar el mejor K según Silhouette
if silhouettes:
    best_k_silhouette_idx = np.argmax(silhouettes)
    best_k_silhouette = rango_k[best_k_silhouette_idx]
else:
    best_k_silhouette = 3 # Fallback si no se pudieron calcular los scores

print(f"Mejor K según Silhouette para el corpus FakeNews: {best_k_silhouette}")

In [ ]:
# 4. Aplicar K-Means con el K seleccionado
K_selected = 8
print(f"Usando K={K_selected} para el clustering final.\n")

kmeans_final = KMeans(n_clusters=K_selected, random_state=42, n_init=10)
clusters_final = kmeans_final.fit_predict(X_df_normalized)

# 5. Visualización con PCA
pca_final = PCA(n_components=2, random_state=42)
X_2d_final = pca_final.fit_transform(X_df_normalized.toarray())

fig_pca, ax_pca = plt.subplots(figsize=(10, 8))

# Generar colores únicos para cada cluster
unique_clusters = np.unique(clusters_final)
num_unique_clusters = len(unique_clusters)
colors_map = plt.get_cmap('viridis', num_unique_clusters) # Usar plt.get_cmap

for i, cluster_id in enumerate(unique_clusters):
    mask = (clusters_final == cluster_id)
    ax_pca.scatter(X_2d_final[mask, 0], X_2d_final[mask, 1],
                   color=colors_map(i), label=f'Cluster {cluster_id}', s=50, alpha=0.7)

ax_pca.set_xlabel(f'PC1 ({pca_final.explained_variance_ratio_[0]:.1%} varianza)')
ax_pca.set_ylabel(f'PC2 ({pca_final.explained_variance_ratio_[1]:.1%} varianza)')
ax_pca.set_title(f'Clusters de documentos (PCA 2D - K-Means, K={K_selected}) - Corpus FakeNews')
ax_pca.legend()
ax_pca.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Mostrar algunas palabras clave por cluster para entenderlos
terminos_df = tfidf_vectorizer.get_feature_names_out()
print("Palabras clave por cluster (K-Means - Corpus FakeNews):\n")
for k_val in range(K_selected):
    if k_val < len(kmeans_final.cluster_centers_):
        centroide = kmeans_final.cluster_centers_[k_val]
        indices_top = centroide.argsort()[::-1][:10] # Top 10 palabras
        palabras_top = [terminos_df[i] for i in indices_top]
        print(f"  Cluster {k_val}: {', '.join(palabras_top)}")
    else:
        print(f"  Cluster {k_val}: No se encontraron documentos para este cluster.")

Pruebe DBSCAN o clustering aglomerativo y compare los resultados

## Resumen

En este capítulo aprendimos:

- **Clustering**: Técnica no supervisada para agrupar documentos similares.
- **K-Means**: Algoritmo iterativo que asigna documentos al centroide más cercano.
- **Selección de K**: Método del codo y coeficiente de Silhouette.
- **Visualización**: PCA para proyectar clusters en 2D.
- **Interpretación**: Palabras clave de los centroides para entender cada grupo.

En el próximo capítulo pasaremos al aprendizaje **supervisado** con la **clasificación de textos**.